# SFT Distillation: Training Qwen3.5 on Claude Reasoning Traces

**Author:** Behrooz Azarkhalili  
**Last updated:** 2026-04

---

## What You'll Learn

This notebook demonstrates **reasoning distillation** via Supervised Fine-Tuning (SFT) --
transferring the chain-of-thought reasoning capabilities of large frontier models
(Claude Opus 4 / Sonnet 4) into a compact, locally-deployable Qwen3.5 model.

| Step | Description |
|------|-------------|
| 1. Dataset | Load Claude reasoning traces (`ermiaazarkhalili/claude-reasoning-distillation`) |
| 2. Adapter | Map the `thinking` field to Qwen3.5's native `reasoning_content` format |
| 3. Training | QLoRA fine-tuning with TRL's `SFTTrainer` |
| 4. Inference | Generate reasoning traces with the trained model |
| 5. Export | Merge adapter, convert to GGUF, push to Hub |

### Hardware Requirements

- **Minimum:** 16 GB VRAM (e.g. NVIDIA T4, RTX 4090, A10G)
- **Recommended:** 24+ GB VRAM for the 2B variant
- **SLURM:** Works on any partition with at least one GPU

### Key Benefits

- **Reasoning distillation** from Claude Opus/Sonnet into a tiny open model
- **100% thinking traces** -- every training sample includes chain-of-thought
- **Native Qwen3.5 thinking** via `enable_thinking=True` at inference
- **QLoRA** keeps memory footprint under 10 GB for the 0.8B model

In [ ]:
# ============================================================================
# Install Dependencies
# ============================================================================
# Colab / bare-metal -- uncomment the pip line below:
!pip install -q transformers trl peft accelerate bitsandbytes datasets

# SLURM (Digital Research Alliance / Fir cluster):
#   module load gcc arrow python/3.11.5
#   source /scratch/$USER/venvs/hf_trl/bin/activate
# All packages are pre-installed in the venv -- skip the pip install above.

In [ ]:
from __future__ import annotations

import os
import warnings
from dataclasses import dataclass, field
from typing import Any

import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

warnings.filterwarnings("ignore", category=FutureWarning)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ---------------------------------------------------------------------------
# GPU detection
# ---------------------------------------------------------------------------
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"✅ GPU detected: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("⚠️ No GPU detected -- training will be extremely slow")

print(f"✅ PyTorch {torch.__version__}")
print(f"✅ CUDA available: {torch.cuda.is_available()}")
print(f"✅ bfloat16 support: {torch.cuda.is_bf16_supported() if torch.cuda.is_available() else 'N/A'}")

In [ ]:
# ============================================================================
# Hugging Face Authentication
# ============================================================================
# The dataset is private -- you need a valid HF token with read access.
# Option A: set HF_TOKEN env var (recommended for SLURM / CI)
# Option B: run `huggingface-cli login` before launching this notebook
# Option C: use a .env file (uncomment below)

# from dotenv import load_dotenv; load_dotenv()

from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token)
    print("✅ Logged in via HF_TOKEN env var")
else:
    try:
        login()
        print("✅ Logged in via cached credentials")
    except Exception:
        print("⚠️ Not logged in -- run `huggingface-cli login` or set HF_TOKEN")

---

## 1. Configuration

All hyperparameters and paths are centralized in two dataclasses below.
Set `SMOKE_TEST = False` for a quick 5-step validation run before committing to a full training job.

In [ ]:
# ============================================================================
# Smoke Test Flag
# ============================================================================
# Set True for a quick validation run (5 steps), False for full training.
SMOKE_TEST: bool = False


@dataclass
class ModelConfig:
    """Model and quantization settings."""

    model_name: str = "Qwen/Qwen3.5-0.8B"
    # Alternative: "Qwen/Qwen3.5-2B"  (requires 24+ GB VRAM with QLoRA)
    hub_model_id: str = "ermiaazarkhalili/Qwen3.5-0.8B-SFT-Claude-Opus-Reasoning"
    max_length: int = 2048
    load_in_4bit: bool = True
    bnb_4bit_quant_type: str = "nf4"
    use_double_quant: bool = True


@dataclass
class TrainingConfig:
    """Training hyperparameters."""

    output_dir: str = "./output/qwen35-sft-claude-reasoning"
    num_train_epochs: int = 3
    per_device_train_batch_size: int = 4
    gradient_accumulation_steps: int = 4
    learning_rate: float = 2e-4
    lr_scheduler_type: str = "cosine"
    warmup_ratio: float = 0.1
    weight_decay: float = 0.01
    logging_steps: int = 10
    save_strategy: str = "steps"
    save_steps: int = 200
    eval_strategy: str = "steps"
    eval_steps: int = 200
    bf16: bool = True
    gradient_checkpointing: bool = True
    push_to_hub: bool = True
    report_to: str = "none"
    max_steps: int = -1  # overridden by SMOKE_TEST


model_cfg = ModelConfig()
train_cfg = TrainingConfig()

# Override for smoke test
if SMOKE_TEST:
    train_cfg.max_steps = 5
    train_cfg.logging_steps = 1
    train_cfg.save_strategy = "no"
    train_cfg.eval_strategy = "no"
    train_cfg.push_to_hub = False
    print("🚨 max_steps=5, saving/eval disabled")
else:
    print(f"🚀 Training for {train_cfg.num_train_epochs} epochs")

print(f"✅ Model: {model_cfg.model_name}")
print(f"✅ Hub ID: {model_cfg.hub_model_id}")

In [ ]:
def setup_hardware_config() -> tuple[torch.dtype, str]:
    """Detect hardware capabilities and return optimal dtype + attention impl.

    Returns:
        Tuple of (compute_dtype, attn_implementation).
    """
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported():
        compute_dtype = torch.bfloat16
        print("✅ Using bfloat16 compute dtype")
    else:
        compute_dtype = torch.float16
        print("✅ Using float16 compute dtype (bf16 not supported)")

    # Flash Attention 2 requires Ampere+ (sm_80+)
    attn_impl = "eager"
    if torch.cuda.is_available():
        capability = torch.cuda.get_device_capability()
        if capability[0] >= 8:
            try:
                import flash_attn  # noqa: F401
                attn_impl = "flash_attention_2"
                print(f"✅ Flash Attention 2 enabled (sm_{capability[0]}{capability[1]})")
            except ImportError:
                print("✅ flash-attn not installed -- using eager attention")
        else:
            print(f"✅ GPU capability sm_{capability[0]}{capability[1]} < sm_80 -- using eager attention")

    return compute_dtype, attn_impl


compute_dtype, attn_implementation = setup_hardware_config()

---

## 2. Dataset: Claude Reasoning Distillation

The dataset (`ermiaazarkhalili/claude-reasoning-distillation`, config `sft`) contains
10,477 training and 549 test samples of Claude Opus/Sonnet reasoning traces across
coding, math, science, logic, and general-knowledge domains.

### Schema

Each sample has a `messages` list in standard chat format:

```json
[
  {"role": "user", "content": "What is 15 + 27?"},
  {"role": "assistant", "content": "42", "thinking": "Let me add 15 and 27 step by step..."}
]
```

The `thinking` field in assistant turns captures the **full chain-of-thought reasoning**
produced by Claude before generating its final answer. Every sample has a thinking trace
(100% coverage).

In [ ]:
# ============================================================================
# Load Dataset
# ============================================================================
DATASET_NAME = "ermiaazarkhalili/claude-reasoning-distillation"
DATASET_CONFIG = "sft"

ds = load_dataset(DATASET_NAME, DATASET_CONFIG)
train_ds = ds["train"]
test_ds = ds["test"]

print(f"✅ Dataset loaded: {DATASET_NAME} (config={DATASET_CONFIG})")
print(f"     Train: {len(train_ds):,} samples")
print(f"     Test:  {len(test_ds):,} samples")

# ---------------------------------------------------------------------------
# Stats: thinking coverage + domain distribution
# ---------------------------------------------------------------------------
from collections import Counter

thinking_count = 0
domain_counts: Counter[str] = Counter()

for sample in train_ds:
    has_thinking = any(
        msg.get("thinking") for msg in sample["messages"] if msg["role"] == "assistant"
    )
    if has_thinking:
        thinking_count += 1
    # Domain is stored at sample level if available
    if "domain" in sample:
        domain_counts[sample["domain"]] += 1

pct = thinking_count / len(train_ds) * 100
print(f"\n✅ Thinking coverage: {thinking_count:,}/{len(train_ds):,} ({pct:.0f}%)")

if domain_counts:
    print("\n     Domain distribution:")
    for domain, count in domain_counts.most_common():
        print(f"       {domain:>12s}: {count:,} ({count/len(train_ds)*100:.1f}%)")

# ---------------------------------------------------------------------------
# Preview one sample
# ---------------------------------------------------------------------------
sample = train_ds[0]
print("\n--- Sample Preview ---")
for msg in sample["messages"]:
    role = msg["role"].upper()
    content = msg["content"][:200] + "..." if len(msg["content"]) > 200 else msg["content"]
    print(f"[{role}] {content}")
    if msg.get("thinking"):
        thinking_preview = msg["thinking"][:150] + "..." if len(msg["thinking"]) > 150 else msg["thinking"]
        print(f"  [THINKING] {thinking_preview}")

---

## 3. Thinking-Field Adapter: Qwen3.5

Qwen3.5 uses `reasoning_content` (not `thinking`) for its native chain-of-thought format.
When `apply_chat_template(enable_thinking=True)` is called, the tokenizer wraps
`reasoning_content` in `<think>...</think>` tags automatically.

**The adapter does two things:**

1. **Renames** `thinking` to `reasoning_content` in assistant message dicts
2. **Applies** Qwen3.5's chat template with `enable_thinking=True`

This produces a single `text` column containing the fully-formatted training input
with thinking blocks properly encoded.

In [ ]:
def adapt_for_qwen35(sample: dict[str, Any], tokenizer: AutoTokenizer) -> dict[str, str]:
    """Convert dataset messages to Qwen3.5 format with thinking traces.

    Renames 'thinking' -> 'reasoning_content' for assistant turns,
    then applies the Qwen3.5 chat template with thinking enabled.

    Args:
        sample: A dataset row with a 'messages' column.
        tokenizer: The Qwen3.5 tokenizer.

    Returns:
        Dict with a 'text' key containing the formatted training string.
    """
    messages: list[dict[str, str]] = []
    for msg in sample["messages"]:
        m = dict(msg)
        if m["role"] == "assistant" and m.get("thinking"):
            m["reasoning_content"] = m.pop("thinking")
        else:
            m.pop("thinking", None)
        messages.append(m)

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=True,
    )
    return {"text": text}


print("✅ adapt_for_qwen35() defined")

In [ ]:
# ============================================================================
# Load Tokenizer + Apply Adapter
# ============================================================================
tokenizer = AutoTokenizer.from_pretrained(
    model_cfg.model_name,
    trust_remote_code=True,
)

# Ensure pad token is set (Qwen3.5 uses <|endoftext|> as both eos and pad)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    print(f"✅ Set pad_token = eos_token ({tokenizer.pad_token})")

print(f"✅ Tokenizer loaded: {model_cfg.model_name}")
print(f"     Vocab size: {tokenizer.vocab_size:,}")

# Apply adapter to both splits
print("\n⏳ Applying Qwen3.5 adapter to dataset...")
train_ds_adapted = train_ds.map(
    lambda x: adapt_for_qwen35(x, tokenizer),
    desc="Adapting train split",
)
test_ds_adapted = test_ds.map(
    lambda x: adapt_for_qwen35(x, tokenizer),
    desc="Adapting test split",
)
print(f"✅ Adapter applied -- 'text' column created")

# ---------------------------------------------------------------------------
# Before / After preview
# ---------------------------------------------------------------------------
print("\n--- Before (raw messages) ---")
raw = train_ds[0]["messages"]
for msg in raw[:2]:
    role = msg["role"]
    print(f"  [{role}] content={msg['content'][:80]}...")
    if msg.get("thinking"):
        print(f"         thinking={msg['thinking'][:80]}...")

print("\n--- After (formatted text) ---")
formatted = train_ds_adapted[0]["text"]
print(formatted[:500] + "...\n")

# ---------------------------------------------------------------------------
# Token length statistics
# ---------------------------------------------------------------------------
import numpy as np

lengths = [len(tokenizer.encode(t["text"])) for t in train_ds_adapted]
lengths_arr = np.array(lengths)
print(f"✅ Token length stats (train):")
print(f"     Mean:   {lengths_arr.mean():.0f}")
print(f"     Median: {np.median(lengths_arr):.0f}")
print(f"     P95:    {np.percentile(lengths_arr, 95):.0f}")
print(f"     Max:    {lengths_arr.max()}")
print(f"     > {model_cfg.max_length} tokens: {(lengths_arr > model_cfg.max_length).sum()} samples")

---

## 4. Model & Training

We use **QLoRA** (4-bit NormalFloat quantization + LoRA adapters) to fine-tune
the model with minimal memory overhead. The key components:

1. **BitsAndBytes** -- 4-bit quantization of the base model weights
2. **LoRA** -- low-rank adapters on attention + MLP projection layers
3. **SFTTrainer** (TRL 1.0) -- handles tokenization, packing, and training loop

In [ ]:
# ============================================================================
# Model Selection
# ============================================================================
# Choose your base model. The 0.8B variant fits comfortably in 16 GB VRAM.
MODEL_NAME: str = model_cfg.model_name

# Alternatives (uncomment to switch):
# MODEL_NAME = "Qwen/Qwen3.5-2B"   # Better quality, needs 24+ GB VRAM

print(f"✅ Selected model: {MODEL_NAME}")

In [ ]:
def create_qlora_model(
    model_name: str,
    compute_dtype: torch.dtype,
    attn_impl: str,
) -> AutoModelForCausalLM:
    """Load a model with 4-bit QLoRA quantization.

    Args:
        model_name: HuggingFace model identifier.
        compute_dtype: Compute dtype (bfloat16 or float16).
        attn_impl: Attention implementation ('flash_attention_2' or 'eager').

    Returns:
        Quantized model ready for LoRA adapter attachment.
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=model_cfg.load_in_4bit,
        bnb_4bit_quant_type=model_cfg.bnb_4bit_quant_type,
        bnb_4bit_compute_dtype=compute_dtype,
        bnb_4bit_use_double_quant=model_cfg.use_double_quant,
    )

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=bnb_config,
        device_map="auto",
        attn_implementation=attn_impl,
        trust_remote_code=True,
    )

    # Prepare for k-bit training (freeze base, enable gradient for adapters)
    model = prepare_model_for_kbit_training(model)

    param_count = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✅ Model loaded: {model_name}")
    print(f"     Total params:     {param_count:,}")
    print(f"     Trainable params: {trainable:,}")
    print(f"     Quantization:     4-bit NF4 (double_quant={model_cfg.use_double_quant})")
    return model


model = create_qlora_model(MODEL_NAME, compute_dtype, attn_implementation)

In [ ]:
def create_lora_config() -> LoraConfig:
    """Create LoRA configuration targeting attention + MLP layers.

    Returns:
        LoraConfig for Qwen3.5 architecture.
    """
    config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "down_proj",
            "up_proj",
        ],
        bias="none",
        task_type="CAUSAL_LM",
    )
    print(f"✅ LoRA config: r={config.r}, alpha={config.lora_alpha}")
    print(f"     Target modules: {config.target_modules}")
    return config


lora_config = create_lora_config()

In [ ]:
def train_model(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    lora_config: LoraConfig,
    train_dataset: Any,
    eval_dataset: Any,
) -> SFTTrainer:
    """Configure and run SFT training.

    Args:
        model: The quantized base model.
        tokenizer: Tokenizer with pad token set.
        lora_config: LoRA adapter configuration.
        train_dataset: Preprocessed training dataset with 'text' column.
        eval_dataset: Preprocessed evaluation dataset with 'text' column.

    Returns:
        The trained SFTTrainer instance.
    """
    sft_config = SFTConfig(
        output_dir=train_cfg.output_dir,
        num_train_epochs=train_cfg.num_train_epochs,
        per_device_train_batch_size=train_cfg.per_device_train_batch_size,
        per_device_eval_batch_size=train_cfg.per_device_train_batch_size,
        gradient_accumulation_steps=train_cfg.gradient_accumulation_steps,
        learning_rate=train_cfg.learning_rate,
        lr_scheduler_type=train_cfg.lr_scheduler_type,
        warmup_ratio=train_cfg.warmup_ratio,
        weight_decay=train_cfg.weight_decay,
        logging_steps=train_cfg.logging_steps,
        save_strategy=train_cfg.save_strategy,
        save_steps=train_cfg.save_steps,
        eval_strategy=train_cfg.eval_strategy,
        eval_steps=train_cfg.eval_steps,
        bf16=train_cfg.bf16,
        gradient_checkpointing=train_cfg.gradient_checkpointing,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=model_cfg.max_length,
        dataset_text_field="text",
        push_to_hub=train_cfg.push_to_hub,
        hub_model_id=model_cfg.hub_model_id if train_cfg.push_to_hub else None,
        report_to=train_cfg.report_to,
        max_steps=train_cfg.max_steps,
        packing=False,
    )

    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
        peft_config=lora_config,
    )

    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in trainer.model.parameters())
    print(f"✅ Trainer ready")
    print(f"     Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
    print(f"     Effective batch: {train_cfg.per_device_train_batch_size * train_cfg.gradient_accumulation_steps}")
    print(f"\n⏳ Starting training...")

    trainer.train()

    print(f"✅ Training complete!")
    return trainer


# ---------------------------------------------------------------------------
# Execute training
# ---------------------------------------------------------------------------
trainer = train_model(model, tokenizer, lora_config, train_ds_adapted, test_ds_adapted)

---

## 5. Inference: Testing Reasoning Traces

Now let's test the trained model! We'll generate responses with `enable_thinking=True`
to see if the model produces chain-of-thought reasoning in Qwen3.5's native
`<think>...</think>` format.

In [ ]:
# ============================================================================
# Inference with Trained Model
# ============================================================================

def generate_response(
    model: AutoModelForCausalLM,
    tokenizer: AutoTokenizer,
    prompt: str,
    max_new_tokens: int = 1024,
) -> str:
    """Generate a response with thinking enabled.

    Args:
        model: The fine-tuned model (or base + adapter).
        tokenizer: Corresponding tokenizer.
        prompt: User prompt text.
        max_new_tokens: Maximum tokens to generate.

    Returns:
        The generated text (including think tags if present).
    """
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=True,
    )
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )

    # Decode only the generated tokens (skip the prompt)
    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=False)


# ---------------------------------------------------------------------------
# Test prompts
# ---------------------------------------------------------------------------
test_prompts = [
    "What is 15 + 27?",
    "Explain why the sky is blue.",
    "Write a Python function to check if a number is prime.",
]

print("⏳ Generating responses with thinking enabled...\n")

for i, prompt in enumerate(test_prompts, 1):
    print(f"{'='*70}")
    print(f"Prompt {i}: {prompt}")
    print(f"{'='*70}")
    response = generate_response(trainer.model, tokenizer, prompt, max_new_tokens=512)
    print(response)
    print()

---

## 6. GGUF Conversion & Hub Push

Merge the LoRA adapter back into the base model, save the full-precision weights,
and optionally convert to GGUF format for local inference with llama.cpp / Ollama.

In [ ]:
# ============================================================================
# Merge Adapter + Save + GGUF Conversion
# ============================================================================
import gc
from pathlib import Path

MERGED_DIR = Path(train_cfg.output_dir) / "merged"
GGUF_DIR = Path(train_cfg.output_dir) / "gguf"

# Step 1: Save the adapter (already done by trainer, but explicit is better)
adapter_dir = Path(train_cfg.output_dir) / "adapter"
trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"✅ Adapter saved to {adapter_dir}")

# Step 2: Reload base model in float16 and merge
print("\n⏳ Reloading base model for merge...")
del model
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)

merged_model = PeftModel.from_pretrained(base_model, str(adapter_dir))
merged_model = merged_model.merge_and_unload()
print("✅ Adapter merged into base model")

# Step 3: Save merged model
merged_model.save_pretrained(str(MERGED_DIR))
tokenizer.save_pretrained(str(MERGED_DIR))
print(f"✅ Merged model saved to {MERGED_DIR}")

# Step 4: GGUF conversion (optional -- requires llama.cpp convert script)
print("\n⏳ Attempting GGUF conversion...")
try:
    import subprocess
    GGUF_DIR.mkdir(parents=True, exist_ok=True)

    result = subprocess.run(
        [
            "python", "-m", "llama_cpp.convert",
            "--outfile", str(GGUF_DIR / "model-q4_k_m.gguf"),
            "--outtype", "q4_k_m",
            str(MERGED_DIR),
        ],
        capture_output=True,
        text=True,
        timeout=600,
    )
    if result.returncode == 0:
        print(f"✅ GGUF saved to {GGUF_DIR / 'model-q4_k_m.gguf'}")
    else:
        print(f"⚠️ GGUF conversion failed: {result.stderr[:200]}")
        print("     Install llama-cpp-python or use llama.cpp's convert_hf_to_gguf.py manually")
except Exception as e:
    print(f"⚠️ GGUF conversion skipped: {e}")
    print("     You can convert manually later with llama.cpp")

# Step 5: Push to Hub
if train_cfg.push_to_hub:
    print(f"\n⏳ Pushing to Hub: {model_cfg.hub_model_id}")
    merged_model.push_to_hub(model_cfg.hub_model_id)
    tokenizer.push_to_hub(model_cfg.hub_model_id)
    print(f"✅ Model pushed to https://huggingface.co/{model_cfg.hub_model_id}")
else:
    print(f"\n✅ push_to_hub=False -- skipping Hub upload")
    print(f"     To push manually: huggingface-cli upload {model_cfg.hub_model_id} {MERGED_DIR}")

---

## Conclusion

You have successfully distilled Claude's reasoning capabilities into a Qwen3.5 model!

### What We Did

1. **Loaded** 10,477 Claude Opus/Sonnet reasoning traces from the distillation dataset
2. **Adapted** the `thinking` field to Qwen3.5's native `reasoning_content` format
3. **Fine-tuned** with QLoRA (4-bit quantization + LoRA adapters) using TRL's `SFTTrainer`
4. **Tested** inference with `enable_thinking=True` to verify reasoning trace generation
5. **Exported** the merged model (and optionally GGUF) for deployment

### Next Steps

- **Scale up:** Try `Qwen/Qwen3.5-2B` for better reasoning quality
- **GRPO training:** Use the `grpo` config of the same dataset for reinforcement learning
- **Evaluation:** Run `lm-eval` benchmarks (MMLU, GSM8K, HumanEval) to measure gains
- **Deployment:** Convert to GGUF and serve locally with Ollama or llama.cpp
- **Iterate:** Adjust `max_length`, LoRA rank, or learning rate based on eval results

### Resources

- [TRL Documentation](https://huggingface.co/docs/trl)
- [Qwen3.5 Model Card](https://huggingface.co/Qwen/Qwen3.5-0.8B)
- [PEFT / LoRA Guide](https://huggingface.co/docs/peft)
- [QLoRA Paper](https://arxiv.org/abs/2305.14314)
- [Knowledge Distillation Survey](https://arxiv.org/abs/2006.05525)